# Kalaris STT v0: train on a free Kaggle GPU

Fine-tunes NVIDIA Parakeet TDT 0.6B v3 for Hindi, Hinglish and English, using open data the owner approved on 2026-11-08: MUCS 2021 Hinglish, FLEURS Hindi and LibriSpeech.

**Before running:** in the right-hand panel, set **Accelerator** to a GPU (T4 or P100) and turn **Internet** on. Internet needs a phone-verified Kaggle account. Then choose **Save Version → Save & Run All**, so the run continues after you close the tab.

**Time:** Kaggle stops a session at 12 hours. The notebook budgets about 11 and stops cleanly. To continue in a new session, add this notebook's output as an input and set `RESUME` in the settings cell.

**Outputs** (in `/kaggle/working/v0`): `head.nemo`, `full.nemo`, the tokenizer, `dev_scores.json`, and `checkpoints_full/last.ckpt` for resuming.

Generated by `training/kaggle/build_notebook.py`. Edit the repo files, not this notebook.

In [ ]:
import os, time, json, shutil, subprocess, sys
from pathlib import Path

START = time.time()
BUDGET_HOURS = 11.0          # Kaggle stops at 12; this leaves time to save
RESUME = None                # e.g. "/kaggle/input/<previous-output>/v0/checkpoints_full/last.ckpt"
SKIP_HEAD = False            # True when resuming a full stage from a saved head.nemo
HEAD_FROM = None             # e.g. "/kaggle/input/<previous-output>/v0/head.nemo"

REPO = Path("/kaggle/working/kalaris-stt")
DATA = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp/kalaris")
DATA.mkdir(parents=True, exist_ok=True)

def hours_left():
    return BUDGET_HOURS - (time.time() - START) / 3600

def sh(command):
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True, cwd=REPO)

print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)
print(subprocess.run(f"df -h {DATA} /kaggle/working", shell=True, capture_output=True, text=True).stdout)

In [ ]:
!pip install -q "nemo_toolkit[asr]==3.0.0" "whisper-normalizer>=0.1" pyarrow soundfile pyyaml huggingface_hub
import nemo, torch
print("NeMo", nemo.__version__, "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: set Accelerator to a GPU in the right-hand panel."

## Code from the kalaris-stt repo

In [ ]:
!mkdir -p /kaggle/working/kalaris-stt/configs /kaggle/working/kalaris-stt/stt_eval /kaggle/working/kalaris-stt/stt_eval/backends /kaggle/working/kalaris-stt/training /kaggle/working/kalaris-stt/training/configs

In [ ]:
%%writefile /kaggle/working/kalaris-stt/stt_eval/__init__.py
"""Evaluation harness for the Kalaris Labs STT model."""

In [ ]:
%%writefile /kaggle/working/kalaris-stt/stt_eval/manifest.py
"""Clip manifests: JSONL with one clip per line.

Required fields: id, audio_filepath, text, duration, language, dataset, split, license.
speaker is required on training and dev sets, and strongly preferred elsewhere.
lattice is optional: a list of slots, each listing accepted spellings. When it's
present, scoring uses the official OIWER scorer instead of plain WER.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Iterable

REQUIRED_FIELDS = ("id", "audio_filepath", "text", "duration", "language", "dataset", "split", "license")


def read_jsonl(path: str | Path) -> list[dict]:
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def write_jsonl(path: str | Path, rows: Iterable[dict]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


REPO_ROOT = Path(__file__).resolve().parents[1]


def relative_to_repo(path: str | Path) -> str:
    """Path to store in a manifest, so it works on any machine with this repo."""
    return Path(path).resolve().relative_to(REPO_ROOT).as_posix()


def read_manifest(path: str | Path) -> list[dict]:
    """Read and validate a manifest. Relative audio paths resolve against the repo."""
    rows = read_jsonl(path)
    for row in rows:
        if "audio_filepath" in row and not Path(row["audio_filepath"]).is_absolute():
            row["audio_filepath"] = str(REPO_ROOT / row["audio_filepath"])
    seen: set[str] = set()
    for number, row in enumerate(rows, start=1):
        missing = [field for field in REQUIRED_FIELDS if field not in row]
        if missing:
            raise ValueError(f"{path}, line {number}: missing {missing}")
        if row["id"] in seen:
            raise ValueError(f"{path}, line {number}: duplicate id {row['id']}")
        seen.add(row["id"])
    return rows

In [ ]:
%%writefile /kaggle/working/kalaris-stt/stt_eval/hf_audio.py
"""Read audio datasets straight from Hugging Face's parquet files.

This avoids the `datasets` audio decoder, and works for gated repos once
`hf auth login` has been run and the terms accepted.
"""

from __future__ import annotations

import io
import json
import subprocess
import urllib.error
import urllib.request
from pathlib import Path
from typing import Iterator

import numpy as np
import pyarrow.parquet as pq
import soundfile
from huggingface_hub import get_token

SAMPLE_RATE = 16000


def hf_request(url: str) -> urllib.request.Request:
    headers = {"User-Agent": "kalaris-stt"}
    token = get_token()
    if token:
        headers["Authorization"] = f"Bearer {token}"
    return urllib.request.Request(url, headers=headers)


def parquet_urls(repo: str, config: str, split: str) -> list[str]:
    url = f"https://huggingface.co/api/datasets/{repo}/parquet/{config}/{split}"
    try:
        with urllib.request.urlopen(hf_request(url), timeout=60) as response:
            return json.load(response)
    except urllib.error.HTTPError as error:
        if error.code in (401, 403):
            raise SystemExit(
                f"No access to {repo}. Run `hf auth login`, then accept its terms at "
                f"https://huggingface.co/datasets/{repo}"
            ) from error
        raise


def to_16k_mono(data: bytes) -> np.ndarray:
    """Decode with soundfile when possible, and use ffmpeg for anything else."""
    samples, rate = soundfile.read(io.BytesIO(data), dtype="float32", always_2d=True)
    if rate == SAMPLE_RATE:
        return samples.mean(axis=1)
    converted = subprocess.run(
        ["ffmpeg", "-loglevel", "error", "-i", "pipe:0", "-ac", "1", "-ar", str(SAMPLE_RATE), "-f", "f32le", "pipe:1"],
        input=data, capture_output=True, check=True,
    ).stdout
    return np.frombuffer(converted, dtype=np.float32)


def iter_records(repo: str, config: str, split: str, work_dir: Path) -> Iterator[dict]:
    """Yield rows one parquet file at a time, deleting each file after use."""
    work_dir.mkdir(parents=True, exist_ok=True)
    download = work_dir / "download.parquet"
    for url in parquet_urls(repo, config, split):
        with urllib.request.urlopen(hf_request(url), timeout=120) as response, open(download, "wb") as out:
            while block := response.read(1 << 20):
                out.write(block)
        try:
            for batch in pq.ParquetFile(download).iter_batches(batch_size=32):
                yield from batch.to_pylist()
        finally:
            download.unlink(missing_ok=True)

In [ ]:
%%writefile /kaggle/working/kalaris-stt/stt_eval/normalize.py
"""Scoring-text normalization for English, Hindi and Hinglish.

Scoring text is not display text. These functions exist so that two models are
never ranked differently because of punctuation or Unicode spelling.

Choices that move Hindi WER are parameters, and reports must state them.
"""

from __future__ import annotations

import re
import unicodedata

DEVANAGARI_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")
NUKTA = "़"
ZERO_WIDTH = dict.fromkeys(map(ord, "​‌‍﻿"))


def is_devanagari_char(ch: str) -> bool:
    return "ऀ" <= ch <= "ॿ" or "꣠" <= ch <= "ꣿ"


def _strip_punctuation(text: str) -> str:
    # Keep letters, digits and combining marks. Devanagari vowel signs are
    # category Mc/Mn, so a plain "\w" rule would split words apart.
    kept = []
    for ch in text:
        category = unicodedata.category(ch)
        if category[0] in "LN" or category in ("Mn", "Mc", "Me"):
            kept.append(ch)
        else:
            kept.append(" ")
    return "".join(kept)


def normalize_hi(text: str, *, fold_nukta: bool = False, fold_nasal: bool = False) -> str:
    """Normalize Hindi or mixed Hindi-English text.

    Args:
        fold_nukta: Treat ज़ and ज as the same letter. Hindi writers use nukta
            inconsistently, so strict WER can penalize correct speech.
        fold_nasal: Treat candrabindu (ँ) and anusvara (ं) as the same letter.
    """
    # NFC is required, not cosmetic. U+0958 (क़) is a composition exclusion, so
    # NFC always yields the two-codepoint form and both spellings compare equal.
    text = unicodedata.normalize("NFC", text)
    text = text.translate(ZERO_WIDTH)
    text = text.translate(DEVANAGARI_DIGITS)
    text = _strip_punctuation(text).lower()
    if fold_nukta:
        text = text.replace(NUKTA, "")
    if fold_nasal:
        text = text.replace("ँ", "ं")
    return " ".join(text.split())


def normalize_en(text: str) -> str:
    """Normalize English with the Whisper normalizer used by leaderboards."""
    from whisper_normalizer.english import EnglishTextNormalizer

    return EnglishTextNormalizer()(text)


def normalize(text: str, language: str, **options: bool) -> str:
    if language == "en":
        return normalize_en(text)
    if language in ("hi", "hinglish"):
        return normalize_hi(text, **options)
    raise ValueError(f"No normalizer for language: {language}")


# Consonant map for the script-insensitive key. Vowels are dropped on purpose,
# since Hindi loanword spelling varies mostly in its vowels.
_DEVANAGARI_CONSONANTS = {
    "क": "k", "ख": "k", "ग": "g", "घ": "g", "ङ": "n",
    "च": "c", "छ": "c", "ज": "j", "झ": "j", "ञ": "n",
    "ट": "t", "ठ": "t", "ड": "d", "ढ": "d", "ण": "n",
    "त": "t", "थ": "t", "द": "d", "ध": "d", "न": "n",
    "प": "p", "फ": "f", "ब": "b", "भ": "b", "म": "n",
    "र": "r", "ल": "l", "ळ": "l", "स": "s", "श": "s", "ष": "s",
    "ं": "n", "ँ": "n",
}
_LATIN_DIGRAPHS = (("ph", "f"), ("ch", "c"), ("sh", "s"), ("ck", "k"), ("th", "t"), ("dh", "d"))
_LATIN_CONSONANTS = {
    "b": "b", "c": "k", "d": "d", "f": "f", "g": "g", "j": "j", "k": "k",
    "l": "l", "m": "n", "n": "n", "p": "p", "q": "k", "r": "r", "s": "s",
    "t": "t", "v": "b", "x": "k", "z": "j",
}


def script_insensitive_key(word: str) -> str:
    """Map a word to a consonant skeleton shared by Devanagari and Latin.

    Both "computer" and "कंप्यूटर" map to "knptr", so a transcript that writes an
    English word in Devanagari is not counted as wrong.

    This is a heuristic. Unrelated words can collide, so the resulting WER is a
    lower bound on real errors. Report it beside strict WER, never instead of it.
    """
    word = unicodedata.normalize("NFD", word.lower())
    if any(is_devanagari_char(ch) for ch in word):
        skeleton = "".join(_DEVANAGARI_CONSONANTS.get(ch, "") for ch in word)
    else:
        # Soft c sounds like s ("office"), while hard c sounds like k ("computer").
        word = re.sub(r"c(?=[eiy])", "S", word)
        for digraph, value in _LATIN_DIGRAPHS:
            word = word.replace(digraph, value.upper())
        skeleton = "".join(
            ch.lower() if ch.isupper() else _LATIN_CONSONANTS.get(ch, "") for ch in word
        )
    collapsed = re.sub(r"(.)\1+", r"\1", skeleton)
    return collapsed or "_"


def to_script_insensitive(text: str) -> str:
    """Apply the script-insensitive key to every word of normalized text."""
    return " ".join(script_insensitive_key(word) for word in text.split())


def word_script(word: str) -> str:
    if any(is_devanagari_char(ch) for ch in word):
        return "deva"
    if any(ch.isalpha() and ch.isascii() for ch in word):
        return "latn"
    return "other"


def switching_level(raw_text: str) -> str:
    """Classify a reference by how it code-switches, following HiKE's levels.

    Returns:
        "mono": one script only. Romanized Hinglish lands here, and can't be
            detected from script alone.
        "word": single embedded words from the minority script.
        "phrase": runs of two or more minority-script words.
        "sentence": each sentence is one script, and the sentences differ.
    """
    sentences = [s for s in re.split(r"[.?!।॥]+", raw_text) if s.strip()]
    sentence_scripts = []
    for sentence in sentences:
        found = {word_script(w) for w in normalize_hi(sentence).split()} - {"other"}
        sentence_scripts.append(found)

    all_scripts = set().union(*sentence_scripts) if sentence_scripts else set()
    if len(all_scripts) < 2:
        return "mono"
    if all(len(found) == 1 for found in sentence_scripts):
        return "sentence"

    scripts = [word_script(w) for w in normalize_hi(raw_text).split()]
    scripts = [s for s in scripts if s != "other"]
    minority = min(all_scripts, key=scripts.count)
    longest = current = 0
    for script in scripts:
        current = current + 1 if script == minority else 0
        longest = max(longest, current)
    return "word" if longest == 1 else "phrase"

In [ ]:
%%writefile /kaggle/working/kalaris-stt/stt_eval/score.py
"""WER, CER and paired significance testing."""

from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Sequence

import numpy as np


@dataclass(frozen=True)
class Errors:
    substitutions: int = 0
    insertions: int = 0
    deletions: int = 0
    reference_length: int = 0

    @property
    def edits(self) -> int:
        return self.substitutions + self.insertions + self.deletions

    @property
    def rate(self) -> float:
        """Error rate in percent. Undefined for an empty reference."""
        if self.reference_length == 0:
            return float("nan")
        return 100.0 * self.edits / self.reference_length

    def __add__(self, other: "Errors") -> "Errors":
        return Errors(
            self.substitutions + other.substitutions,
            self.insertions + other.insertions,
            self.deletions + other.deletions,
            self.reference_length + other.reference_length,
        )

    def as_dict(self) -> dict:
        return {**asdict(self), "rate": self.rate}


def align(reference: Sequence[str], hypothesis: Sequence[str]) -> Errors:
    """Count edits with Levenshtein alignment.

    Among minimum-edit alignments, this prefers substitutions over paired
    insertions and deletions, matching standard scoring tools.
    """
    # Each cell holds (edits, deletions, insertions) for a prefix pair.
    previous = [(j, 0, j) for j in range(len(hypothesis) + 1)]
    for i, ref_token in enumerate(reference, start=1):
        current = [(i, i, 0)]
        for j, hyp_token in enumerate(hypothesis, start=1):
            diagonal = previous[j - 1]
            cost = 0 if ref_token == hyp_token else 1
            options = (
                (diagonal[0] + cost, diagonal[1], diagonal[2]),
                (previous[j][0] + 1, previous[j][1] + 1, previous[j][2]),
                (current[j - 1][0] + 1, current[j - 1][1], current[j - 1][2] + 1),
            )
            current.append(min(options, key=lambda cell: cell[0]))
        previous = current
    edits, deletions, insertions = previous[-1]
    return Errors(edits - deletions - insertions, insertions, deletions, len(reference))


def word_errors(reference: str, hypothesis: str) -> Errors:
    return align(reference.split(), hypothesis.split())


def char_errors(reference: str, hypothesis: str) -> Errors:
    """CER over characters, with spaces removed so WER and CER stay independent."""
    return align(list(reference.replace(" ", "")), list(hypothesis.replace(" ", "")))


def oiwer_errors(hypothesis: str, lattice: Sequence[Sequence[str]], language: str = "hindi") -> Errors:
    """Orthographically-informed word errors, using the leaderboard's own scorer.

    The scorer normalizes text itself, so pass raw model output. Any spelling in
    a slot counts as correct, and the corpus rate is total edits over total
    reference words.
    """
    from voi_oiwer import oiwer

    result = oiwer(hypothesis=hypothesis, reference_lists=[list(slot) for slot in lattice], input_language=language)
    insertions, deletions, substitutions = result[4]
    return Errors(substitutions, insertions, deletions, result[5])


def total(errors: Sequence[Errors]) -> Errors:
    result = Errors()
    for item in errors:
        result = result + item
    return result


def corpus_wer(references: Sequence[str], hypotheses: Sequence[str]) -> Errors:
    """Corpus WER: total edits over total words, not a mean of per-clip rates."""
    if len(references) != len(hypotheses):
        raise ValueError(f"{len(references)} references but {len(hypotheses)} hypotheses")
    return total([word_errors(r, h) for r, h in zip(references, hypotheses)])


@dataclass(frozen=True)
class BootstrapResult:
    wer_a: float
    wer_b: float
    difference: float  # WER(B) - WER(A), in points. Negative means B is better.
    ci_low: float
    ci_high: float
    resamples: int
    unit: str

    @property
    def excludes_zero(self) -> bool:
        return self.ci_low > 0 or self.ci_high < 0

    def as_dict(self) -> dict:
        return {**asdict(self), "excludes_zero": self.excludes_zero}


def paired_bootstrap(
    errors_a: Sequence[Errors],
    errors_b: Sequence[Errors],
    *,
    groups: Sequence[str] | None = None,
    resamples: int = 2000,
    confidence: float = 0.95,
    seed: int = 0,
    lattice: bool = False,
) -> BootstrapResult:
    """Confidence interval for WER(B) - WER(A) on the same clips.

    This is the repo's promotion gate. Pass speaker IDs as `groups` whenever
    they exist: clips from one speaker aren't independent, and resampling single
    clips then gives intervals that are too narrow.

    Set `lattice` for OIWER. Each model is measured against the spellings it
    matched, so its reference length can legitimately differ on the same clip.
    """
    if len(errors_a) != len(errors_b):
        raise ValueError("Paired bootstrap needs the same clips for both models")
    if not errors_a:
        raise ValueError("Paired bootstrap needs at least one clip")
    if not lattice and any(a.reference_length != b.reference_length for a, b in zip(errors_a, errors_b)):
        raise ValueError("Clips are misaligned: reference lengths differ")

    edits_a = np.array([e.edits for e in errors_a], dtype=np.int64)
    edits_b = np.array([e.edits for e in errors_b], dtype=np.int64)
    lengths_a = np.array([e.reference_length for e in errors_a], dtype=np.int64)
    lengths_b = np.array([e.reference_length for e in errors_b], dtype=np.int64)

    if groups is not None:
        if len(groups) != len(errors_a):
            raise ValueError("Need one group per clip")
        names, inverse = np.unique(np.array(groups), return_inverse=True)
        edits_a = np.bincount(inverse, weights=edits_a, minlength=len(names))
        edits_b = np.bincount(inverse, weights=edits_b, minlength=len(names))
        lengths_a = np.bincount(inverse, weights=lengths_a, minlength=len(names))
        lengths_b = np.bincount(inverse, weights=lengths_b, minlength=len(names))

    rng = np.random.default_rng(seed)
    picks = rng.integers(0, len(lengths_a), size=(resamples, len(lengths_a)))
    words_a, words_b = lengths_a[picks].sum(axis=1), lengths_b[picks].sum(axis=1)
    usable = (words_a > 0) & (words_b > 0)
    diffs = 100.0 * (
        edits_b[picks].sum(axis=1)[usable] / words_b[usable]
        - edits_a[picks].sum(axis=1)[usable] / words_a[usable]
    )
    tail = 100.0 * (1.0 - confidence) / 2.0
    low, high = np.percentile(diffs, [tail, 100.0 - tail])

    wer_a = 100.0 * edits_a.sum() / lengths_a.sum()
    wer_b = 100.0 * edits_b.sum() / lengths_b.sum()
    return BootstrapResult(
        float(wer_a), float(wer_b), float(wer_b - wer_a), float(low), float(high),
        resamples, "group" if groups is not None else "clip",
    )

In [ ]:
%%writefile /kaggle/working/kalaris-stt/stt_eval/bench.py
"""Speed, memory and size measurement.

Download size, peak RAM and speed are reported separately, because a small
download doesn't imply low RAM. Run one model per process, or peak RAM is
polluted by whatever ran earlier in that process.
"""

from __future__ import annotations

import platform
import resource
import subprocess
import sys
from dataclasses import asdict, dataclass
from pathlib import Path


def peak_rss_bytes() -> int:
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    # macOS reports bytes and Linux reports kilobytes.
    return peak if sys.platform == "darwin" else peak * 1024


def directory_size_bytes(path: str | Path) -> int:
    path = Path(path)
    if path.is_file():
        return path.stat().st_size
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file() and not p.is_symlink())


def hardware_summary() -> dict:
    summary = {"machine": platform.machine(), "system": platform.platform()}
    if sys.platform == "darwin":
        def sysctl(name: str) -> str:
            return subprocess.run(
                ["sysctl", "-n", name], capture_output=True, text=True, check=True
            ).stdout.strip()

        summary["cpu"] = sysctl("machdep.cpu.brand_string")
        summary["memory_gb"] = round(int(sysctl("hw.memsize")) / 2**30)
    return summary


@dataclass
class BenchResult:
    backend: str
    model: str
    clips: int
    audio_seconds: float
    load_seconds: float
    transcribe_seconds: float
    latency_p50_seconds: float
    latency_p90_seconds: float
    peak_rss_bytes: int
    accelerator_peak_bytes: int | None
    model_bytes: int | None

    @property
    def rtfx(self) -> float:
        """Audio seconds per wall-clock second, excluding model load."""
        return self.audio_seconds / self.transcribe_seconds if self.transcribe_seconds else 0.0

    def as_dict(self) -> dict:
        return {**asdict(self), "rtfx": self.rtfx, "hardware": hardware_summary()}

In [ ]:
%%writefile /kaggle/working/kalaris-stt/stt_eval/leakage.py
"""Train/test leakage checks, run before any score is reported.

A checksum finds byte-identical audio. A speaker check finds the same voice.
Re-encoded copies of one recording pass both, so also check transcripts.
"""

from __future__ import annotations

import hashlib
from dataclasses import dataclass, field
from pathlib import Path
from typing import Sequence

from .normalize import normalize_hi


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


@dataclass
class LeakageReport:
    duplicate_audio: list[tuple[str, str]] = field(default_factory=list)
    shared_speakers: list[str] = field(default_factory=list)
    shared_transcripts: list[str] = field(default_factory=list)
    clips_without_speaker: int = 0

    @property
    def is_clean(self) -> bool:
        return not (self.duplicate_audio or self.shared_speakers or self.shared_transcripts)

    def as_dict(self) -> dict:
        return {
            "is_clean": self.is_clean,
            "duplicate_audio": self.duplicate_audio[:50],
            "duplicate_audio_count": len(self.duplicate_audio),
            "shared_speakers": self.shared_speakers[:50],
            "shared_speaker_count": len(self.shared_speakers),
            "shared_transcript_count": len(self.shared_transcripts),
            "clips_without_speaker": self.clips_without_speaker,
        }


def check_leakage(
    train: Sequence[dict],
    held_out: Sequence[dict],
    *,
    check_transcripts: bool = True,
    min_transcript_words: int = 6,
) -> LeakageReport:
    """Compare a training manifest with a dev or test manifest.

    Speaker IDs are namespaced by dataset, because "spk1" in two corpora is not
    the same person. Short transcripts are ignored since common phrases repeat
    naturally.
    """
    report = LeakageReport()

    train_audio = {sha256_file(row["audio_filepath"]): row["id"] for row in train}
    for row in held_out:
        match = train_audio.get(sha256_file(row["audio_filepath"]))
        if match is not None:
            report.duplicate_audio.append((match, row["id"]))

    def speakers(rows: Sequence[dict]) -> set[str]:
        return {f"{r['dataset']}:{r['speaker']}" for r in rows if r.get("speaker")}

    report.shared_speakers = sorted(speakers(train) & speakers(held_out))
    report.clips_without_speaker = sum(1 for r in held_out if not r.get("speaker"))

    if check_transcripts:
        def texts(rows: Sequence[dict]) -> set[str]:
            normalized = (normalize_hi(r["text"]) for r in rows)
            return {t for t in normalized if len(t.split()) >= min_transcript_words}

        report.shared_transcripts = sorted(texts(train) & texts(held_out))

    return report

In [ ]:
%%writefile /kaggle/working/kalaris-stt/stt_eval/cli.py
"""Command-line entry point: transcribe, score, compare, leakage, worst."""

from __future__ import annotations

import argparse
import json
import sys
import time
from collections import defaultdict
from pathlib import Path

import numpy as np

from .bench import BenchResult, peak_rss_bytes
from .leakage import check_leakage
from .manifest import read_jsonl, read_manifest, write_jsonl
from .normalize import normalize, switching_level, to_script_insensitive
from .score import char_errors, oiwer_errors, paired_bootstrap, total, word_errors


def _options(args: argparse.Namespace) -> dict:
    options = {}
    if getattr(args, "fold_nukta", False):
        options["fold_nukta"] = True
    if getattr(args, "fold_nasal", False):
        options["fold_nasal"] = True
    return options


def _paired(manifest: list[dict], hypotheses_path: str) -> list[tuple[dict, str]]:
    """Join clips to hypotheses. A missing clip is an error, never a skip,
    because dropping hard clips would silently flatter a model."""
    hypotheses = {row["id"]: row["text"] for row in read_jsonl(hypotheses_path)}
    missing = [row["id"] for row in manifest if row["id"] not in hypotheses]
    if missing:
        raise SystemExit(f"{hypotheses_path} is missing {len(missing)} clips, e.g. {missing[:3]}")
    return [(row, hypotheses[row["id"]]) for row in manifest]


def command_transcribe(args: argparse.Namespace) -> int:
    from .backends import load_backend

    rows = read_manifest(args.manifest)
    if args.limit:
        rows = rows[: args.limit]

    backend = load_backend(args.backend, args.model, args.language)
    started = time.perf_counter()
    backend.load()
    load_seconds = time.perf_counter() - started

    outputs, latencies = [], []
    for index, row in enumerate(rows, start=1):
        started = time.perf_counter()
        text = backend.transcribe(row["audio_filepath"])
        latency = time.perf_counter() - started
        latencies.append(latency)
        outputs.append({"id": row["id"], "text": text, "seconds": round(latency, 4)})
        if index % 200 == 0:
            print(f"{index}/{len(rows)}", file=sys.stderr, flush=True)

    write_jsonl(args.out, outputs)
    result = BenchResult(
        backend=args.backend,
        model=args.model,
        clips=len(rows),
        audio_seconds=sum(row["duration"] for row in rows),
        load_seconds=load_seconds,
        transcribe_seconds=sum(latencies),
        latency_p50_seconds=float(np.percentile(latencies, 50)),
        latency_p90_seconds=float(np.percentile(latencies, 90)),
        peak_rss_bytes=peak_rss_bytes(),
        accelerator_peak_bytes=backend.accelerator_peak_bytes(),
        model_bytes=backend.model_bytes(),
    )
    bench_path = Path(args.out).with_suffix(".bench.json")
    bench_path.write_text(json.dumps(result.as_dict(), indent=2), encoding="utf-8")
    print(json.dumps(result.as_dict(), indent=2))
    return 0


def command_score(args: argparse.Namespace) -> int:
    rows = read_manifest(args.manifest)
    if args.limit:
        rows = rows[: args.limit]
    pairs = _paired(rows, args.hypotheses)
    options = _options(args)

    if any("lattice" in row for row, _ in pairs):
        if not all("lattice" in row for row, _ in pairs):
            raise SystemExit("Manifest mixes clips with and without lattices")
        result = total([oiwer_errors(hypothesis, row["lattice"]) for row, hypothesis in pairs])
        print(json.dumps({
            "manifest": args.manifest,
            "hypotheses": args.hypotheses,
            "language": args.language,
            "metric": "oiwer",
            "clips": len(pairs),
            "wer": result.as_dict(),
        }, indent=2, ensure_ascii=False))
        return 0

    word, char, loose = [], [], []
    by_level: dict[str, list] = defaultdict(list)
    for row, hypothesis in pairs:
        reference_text = normalize(row["text"], args.language, **options)
        hypothesis_text = normalize(hypothesis, args.language, **options)
        errors = word_errors(reference_text, hypothesis_text)
        word.append(errors)
        char.append(char_errors(reference_text, hypothesis_text))
        if args.language == "hinglish":
            loose.append(word_errors(to_script_insensitive(reference_text), to_script_insensitive(hypothesis_text)))
            by_level[switching_level(row["text"])].append(errors)

    report = {
        "manifest": args.manifest,
        "hypotheses": args.hypotheses,
        "language": args.language,
        "normalization": options or "strict",
        "clips": len(pairs),
        "wer": total(word).as_dict(),
        "cer": total(char).as_dict(),
    }
    if loose:
        report["script_insensitive_wer"] = total(loose).as_dict()
        report["wer_by_switching_level"] = {
            level: {"clips": len(items), **total(items).as_dict()} for level, items in sorted(by_level.items())
        }
    print(json.dumps(report, indent=2, ensure_ascii=False))
    return 0


def command_compare(args: argparse.Namespace) -> int:
    rows = read_manifest(args.manifest)
    options = _options(args)

    def errors_for(path: str) -> list:
        return [
            oiwer_errors(text, row["lattice"]) if "lattice" in row
            else word_errors(normalize(row["text"], args.language, **options), normalize(text, args.language, **options))
            for row, text in _paired(rows, path)
        ]

    has_speakers = all(row.get("speaker") for row in rows)
    groups = [f"{row['dataset']}:{row['speaker']}" for row in rows] if has_speakers else None
    lattice = any("lattice" in row for row in rows)
    result = paired_bootstrap(errors_for(args.a), errors_for(args.b), groups=groups, seed=args.seed, lattice=lattice)
    print(json.dumps({"a": args.a, "b": args.b, "language": args.language, **result.as_dict()}, indent=2))
    return 0


def command_leakage(args: argparse.Namespace) -> int:
    report = check_leakage(read_manifest(args.train), read_manifest(args.held_out))
    print(json.dumps(report.as_dict(), indent=2, ensure_ascii=False))
    return 0 if report.is_clean else 1


def command_worst(args: argparse.Namespace) -> int:
    """List the worst clips for error analysis. Refuses test sets by design."""
    rows = read_manifest(args.manifest)
    test_clips = sum(1 for row in rows if row["split"] == "test")
    if test_clips:
        print(
            f"Refused: {args.manifest} has {test_clips} test clips. "
            "Per-clip test results would let tuning leak into the test set.",
            file=sys.stderr,
        )
        return 2

    scored = []
    for row, hypothesis in _paired(rows, args.hypotheses):
        reference = normalize(row["text"], args.language)
        output = normalize(hypothesis, args.language)
        errors = word_errors(reference, output)
        scored.append((errors.edits, row["id"], reference, output))
    scored.sort(key=lambda item: item[0], reverse=True)
    for edits, clip_id, reference, output in scored[: args.count]:
        print(json.dumps({"id": clip_id, "edits": edits, "ref": reference, "hyp": output}, ensure_ascii=False))
    return 0


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(prog="stt-eval", description=__doc__)
    commands = parser.add_subparsers(dest="command", required=True)

    def add_normalization(sub: argparse.ArgumentParser) -> None:
        sub.add_argument("--language", required=True, choices=["en", "hi", "hinglish"])
        sub.add_argument("--fold-nukta", action="store_true", help="Treat ज़ and ज as equal")
        sub.add_argument("--fold-nasal", action="store_true", help="Treat ँ and ं as equal")

    transcribe = commands.add_parser("transcribe", help="Run a model and record speed and memory")
    transcribe.add_argument("--backend", required=True)
    transcribe.add_argument("--model", required=True)
    transcribe.add_argument("--manifest", required=True)
    transcribe.add_argument("--out", required=True)
    transcribe.add_argument("--language", help="Language code for backends that need one")
    transcribe.add_argument("--limit", type=int, help="Use only the first N clips")
    transcribe.set_defaults(handler=command_transcribe)

    score = commands.add_parser("score", help="Report aggregate WER and CER")
    score.add_argument("--manifest", required=True)
    score.add_argument("--hypotheses", required=True)
    score.add_argument("--limit", type=int, help="Score only the first N clips")
    add_normalization(score)
    score.set_defaults(handler=command_score)

    compare = commands.add_parser("compare", help="Paired bootstrap between two models")
    compare.add_argument("--manifest", required=True)
    compare.add_argument("--a", required=True, help="Baseline hypotheses")
    compare.add_argument("--b", required=True, help="Candidate hypotheses")
    compare.add_argument("--seed", type=int, default=0)
    add_normalization(compare)
    compare.set_defaults(handler=command_compare)

    leakage = commands.add_parser("leakage", help="Check a held-out set against training data")
    leakage.add_argument("--train", required=True)
    leakage.add_argument("--held-out", required=True)
    leakage.set_defaults(handler=command_leakage)

    worst = commands.add_parser("worst", help="Show worst clips (dev sets only)")
    worst.add_argument("--manifest", required=True)
    worst.add_argument("--hypotheses", required=True)
    worst.add_argument("--language", required=True, choices=["en", "hi", "hinglish"])
    worst.add_argument("--count", type=int, default=20)
    worst.set_defaults(handler=command_worst)

    return parser


def main(argv: list[str] | None = None) -> int:
    args = build_parser().parse_args(argv)
    return args.handler(args)


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile /kaggle/working/kalaris-stt/stt_eval/backends/__init__.py
"""Inference adapters. Each imports its runtime only when it is selected."""

from __future__ import annotations

from .base import Backend


def load_backend(name: str, model: str, language: str | None = None) -> Backend:
    if name == "parakeet-mlx":
        from .parakeet_mlx_backend import ParakeetMLX

        return ParakeetMLX(model)
    if name == "whisper-mlx":
        from .whisper_mlx_backend import WhisperMLX

        return WhisperMLX(model, language)
    if name == "sherpa-onnx-transducer":
        from .sherpa_onnx_backend import SherpaOnnxTransducer

        return SherpaOnnxTransducer(model)
    if name == "indicconformer-onnx":
        from .indicconformer_onnx_backend import IndicConformerOnnx

        return IndicConformerOnnx(model, language)
    if name == "nemo":
        from .nemo_backend import NemoBackend

        return NemoBackend(model)
    raise ValueError(f"Unknown backend: {name}")

In [ ]:
%%writefile /kaggle/working/kalaris-stt/stt_eval/backends/base.py
from __future__ import annotations

from abc import ABC, abstractmethod
from pathlib import Path


class Backend(ABC):
    name = "base"

    def __init__(self, model: str):
        self.model = model

    @abstractmethod
    def load(self) -> None:
        """Load the model. Timed separately from transcription."""

    @abstractmethod
    def transcribe(self, audio_path: str) -> str:
        """Return the transcript of one 16 kHz mono audio file."""

    def accelerator_peak_bytes(self) -> int | None:
        """Peak accelerator memory, when the runtime exposes it."""
        return None

    def model_bytes(self) -> int | None:
        """On-disk model size, when known."""
        return None


def huggingface_cache_bytes(repo_id: str) -> int | None:
    """Size of a downloaded Hugging Face repo, without counting symlinks twice."""
    from huggingface_hub import scan_cache_dir

    for repo in scan_cache_dir().repos:
        if repo.repo_id == repo_id:
            return repo.size_on_disk
    return None


def local_path_bytes(path: str | Path) -> int:
    from ..bench import directory_size_bytes

    return directory_size_bytes(path)

In [ ]:
%%writefile /kaggle/working/kalaris-stt/stt_eval/backends/nemo_backend.py
from __future__ import annotations

from pathlib import Path

from .base import Backend, huggingface_cache_bytes, local_path_bytes


class NemoBackend(Backend):
    """Any NeMo ASR model: a Hugging Face ID or a local .nemo file.

    Run from the training environment (.venv-train), which has NeMo installed.
    """

    name = "nemo"

    def load(self) -> None:
        import torch
        from nemo.collections.asr.models import ASRModel

        if Path(self.model).exists():
            self._model = ASRModel.restore_from(self.model, map_location="cpu")
        else:
            self._model = ASRModel.from_pretrained(self.model, map_location="cpu")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self._model = self._model.to(device).eval()

    def transcribe(self, audio_path: str) -> str:
        output = self._model.transcribe([audio_path], batch_size=1, verbose=False)
        # Depending on the model, NeMo returns a list or a (best, all) pair.
        first = output[0] if not isinstance(output, tuple) else output[0][0]
        if isinstance(first, list):
            first = first[0]
        return getattr(first, "text", first)

    def accelerator_peak_bytes(self) -> int | None:
        import torch

        return torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None

    def model_bytes(self) -> int | None:
        if Path(self.model).exists():
            return local_path_bytes(self.model)
        return huggingface_cache_bytes(self.model)

In [ ]:
%%writefile /kaggle/working/kalaris-stt/training/prepare_data.py
"""Build NeMo training and dev manifests from Hugging Face datasets.

Usage:
    .venv/bin/python training/prepare_data.py training/configs/wave1.yaml

Safeguards:
- Refuses any source whose license isn't `confirmed` in configs/datasets.yaml.
  `--smoke` allows `listed` sources, for pipeline tests whose weights are thrown away.
- Refuses test splits and anything listed in configs/benchmarks.yaml.
- Keeps dev speakers out of training, by hashing each speaker to one side.
"""

from __future__ import annotations

import argparse
import hashlib
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import soundfile
import yaml

from stt_eval.hf_audio import SAMPLE_RATE, iter_records, to_16k_mono
from stt_eval.manifest import REPO_ROOT, write_jsonl

ZERO_WIDTH = dict.fromkeys(map(ord, "​‌‍﻿"))


def clean_text(text: str) -> str:
    """Training text keeps what users should see. It only fixes encoding noise."""
    return " ".join(unicodedata.normalize("NFC", text).translate(ZERO_WIDTH).split())


def rejection_reason(text: str, duration: float, limits: dict) -> str | None:
    if not text:
        return "empty_text"
    if duration < limits["min_seconds"]:
        return "too_short"
    if duration > limits["max_seconds"]:
        return "too_long"
    rate = len(text) / duration
    if rate < limits["min_chars_per_second"] or rate > limits["max_chars_per_second"]:
        return "bad_alignment"
    return None


def is_dev_speaker(dataset: str, speaker: str | None, dev_fraction: float) -> bool:
    """Assign a speaker to one side, stably. Clips with no speaker stay in training."""
    if not speaker or dev_fraction <= 0:
        return False
    digest = hashlib.sha256(f"{dataset}:{speaker}".encode()).digest()
    return int.from_bytes(digest[:8], "big") / 2**64 < dev_fraction


def check_source_allowed(source: dict, registry: dict, benchmarks: dict, *, smoke: bool) -> None:
    name, repo, split = source["name"], source["repo"], source["split"]
    if split.startswith("test"):
        raise SystemExit(f"{name}: refusing to train on test split {split}")

    benchmark_repos = {b["repo"] for b in benchmarks["primary"].values()}
    benchmark_repos |= {b["repo"] for b in benchmarks["english_average"]}
    if repo in benchmark_repos:
        raise SystemExit(f"{name}: {repo} is a benchmark, so it can't be used for training")

    entry = next((d for d in registry["datasets"] if d["name"] == source["registry"]), None)
    if entry is None:
        raise SystemExit(f"{name}: {source['registry']} isn't in configs/datasets.yaml")
    allowed = {"confirmed", "listed"} if smoke else {"confirmed"}
    if entry["status"] not in allowed:
        raise SystemExit(
            f"{name}: {source['registry']} has status '{entry['status']}'. A person must review its "
            "license and set status: confirmed in configs/datasets.yaml."
        )


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("config")
    parser.add_argument("--smoke", action="store_true", help="Allow unreviewed licenses for a throwaway pipeline test")
    parser.add_argument("--max-hours", type=float, help="Cap every source at this many hours, for quick checks")
    parser.add_argument("--output-root", help="Where audio and manifests go (default: data/training in the repo)")
    args = parser.parse_args()

    config = yaml.safe_load(Path(args.config).read_text(encoding="utf-8"))
    registry = yaml.safe_load((REPO_ROOT / "configs" / "datasets.yaml").read_text(encoding="utf-8"))
    benchmarks = yaml.safe_load((REPO_ROOT / "configs" / "benchmarks.yaml").read_text(encoding="utf-8"))
    sources = [s for s in config["sources"] if s.get("enabled", True)]
    for source in sources:
        check_source_allowed(source, registry, benchmarks, smoke=args.smoke)

    output = Path(args.output_root or REPO_ROOT / "data" / "training") / config["name"]
    limits = config["limits"]
    train: list[dict] = []
    dev: dict[str, list[dict]] = defaultdict(list)
    report = []

    for source in sources:
        name = source["name"]
        audio_dir = output / "audio" / name
        rejected: Counter = Counter()
        seconds = 0.0
        kept = 0
        max_seconds = min(source.get("max_hours", float("inf")), args.max_hours or float("inf")) * 3600
        text_column, speaker_column = source["text_column"], source.get("speaker_column")

        for record in iter_records(source["repo"], source.get("config", "default"), source["split"], audio_dir):
            if seconds >= max_seconds:
                break
            samples = to_16k_mono(record["audio"]["bytes"])
            duration = len(samples) / SAMPLE_RATE
            text = clean_text(record[text_column] or "")
            if source.get("lowercase"):
                text = text.lower()
            reason = rejection_reason(text, duration, limits)
            if reason:
                rejected[reason] += 1
                continue

            path = audio_dir / f"{kept:07d}.flac"
            soundfile.write(path, samples, SAMPLE_RATE, format="FLAC")
            speaker = str(record[speaker_column]) if speaker_column and record.get(speaker_column) is not None else None
            row = {
                "id": f"{name}-{kept:07d}",
                "audio_filepath": str(path),
                "text": text,
                "duration": duration,
                "language": source["language"],
                "speaker": speaker,
                "dataset": name,
                "split": "train",
                "license": source["license"],
            }
            # `role: dev` sources are held out whole, such as an official validation split.
            if source.get("role") == "dev" or is_dev_speaker(name, speaker, source.get("dev_fraction", 0.0)):
                dev[source["language"]].append({**row, "split": "dev"})
            else:
                # `repeat` oversamples scarce sources, such as real Hinglish.
                train.extend([row] * source.get("repeat", 1))
            seconds += duration
            kept += 1
        report.append(f"{name}: {kept} clips, {seconds / 3600:.2f} hours, rejected {dict(rejected)}")

    write_jsonl(output / "train.json", train)
    for language, rows in dev.items():
        write_jsonl(output / f"dev_{language}.json", rows)
    write_jsonl(output / "dev.json", [row for rows in dev.values() for row in rows])

    summary = "\n".join(report + [f"train rows: {len(train)}", f"dev rows: { {k: len(v) for k, v in dev.items()} }"])
    (output / "report.txt").write_text(summary + "\n", encoding="utf-8")
    print(summary)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/kalaris-stt/training/train_tokenizer.py
"""Train one SentencePiece BPE tokenizer shared across scripts.

Wave 2 scripts are included from day one by mixing in FLEURS text for those
languages. No audio is needed for this, and it means later languages won't force
a decoder reset.

Usage:
    .venv-train/bin/python training/train_tokenizer.py \
        --manifest data/training/wave1/train.json --output models/tokenizers/wave1 \
        --vocab-size 4096 --extra-fleurs bn_in,mr_in,te_in,ta_in,gu_in,kn_in,ml_in,pa_in,or_in
"""

from __future__ import annotations

import argparse
import csv
import json
import tempfile
import unicodedata
import urllib.request
from collections import Counter
from pathlib import Path

FLEURS_TSV = "https://huggingface.co/datasets/google/fleurs/resolve/main/data/{config}/train.tsv"


def fleurs_text(config: str) -> list[str]:
    request = urllib.request.Request(FLEURS_TSV.format(config=config), headers={"User-Agent": "kalaris-stt"})
    with urllib.request.urlopen(request, timeout=60) as response:
        rows = csv.reader(response.read().decode("utf-8").splitlines(), delimiter="\t", quotechar='"')
        # Columns: id, file name, raw transcription, ...
        return list({row[2] for row in rows if len(row) > 2})


def script_counts(pieces: list[str]) -> Counter:
    counts: Counter = Counter()
    for piece in pieces:
        for character in piece:
            if character.isalpha():
                counts[unicodedata.name(character, "UNKNOWN").split(" ")[0]] += 1
                break
    return counts


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--manifest", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--vocab-size", type=int, default=4096)
    parser.add_argument("--extra-fleurs", default="", help="Comma-separated FLEURS configs for later-wave scripts")
    args = parser.parse_args()

    with open(args.manifest, encoding="utf-8") as handle:
        lines = list({json.loads(line)["text"] for line in handle if line.strip()})
    for config in filter(None, args.extra_fleurs.split(",")):
        lines.extend(fleurs_text(config))

    from nemo.collections.common.tokenizers.sentencepiece_tokenizer import create_spt_model

    output = Path(args.output)
    output.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile("w", suffix=".txt", encoding="utf-8", delete=False) as corpus:
        corpus.write("\n".join(lines))

    create_spt_model(
        data_file=corpus.name,
        vocab_size=args.vocab_size,
        sample_size=-1,
        do_lower_case=False,
        tokenizer_type="bpe",
        output_dir=str(output),
    )
    Path(corpus.name).unlink()

    pieces = (output / "tokenizer.vocab").read_text(encoding="utf-8").splitlines()
    print(f"{len(lines)} text lines, {len(pieces)} tokens -> {output}")
    print("tokens per script:", dict(script_counts([p.split("\t")[0] for p in pieces])))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/kalaris-stt/training/finetune.py
"""Fine-tune a NeMo transducer. Run from the training environment (.venv-train).

Stages run in order, each starting from the last one's .nemo file:
    head: new tokenizer, decoder and joint; encoder frozen
    full: everything trains, with lower learning rates near the input

Usage:
    .venv-train/bin/python training/finetune.py training/configs/smoke.yaml --stage head
    .venv-train/bin/python training/finetune.py training/configs/smoke.yaml --stage full
"""

from __future__ import annotations

import argparse
from pathlib import Path

import lightning.pytorch as pl
import torch
import yaml
from lightning.pytorch.callbacks import Callback, ModelCheckpoint
from nemo.collections.asr.losses.rnnt import RNNTLoss
from nemo.collections.asr.models import ASRModel
from omegaconf import OmegaConf, open_dict


def restore_tdt_loss(model) -> None:
    """Put the TDT loss back after change_vocabulary.

    NeMo 3.0's change_vocabulary always rebuilds a plain RNNTLoss, sized without
    the duration outputs. TDT models would then train on the wrong loss with no
    error, so this rebuilds it the way the model's constructor does.
    """
    print(f"Loss before rebuild: {model.loss._loss.__class__.__name__}, blank {model.loss._loss.blank}")
    loss_name, loss_kwargs = model.extract_rnnt_loss_cfg(model.cfg.get("loss", None))
    num_classes = model.joint.num_classes_with_blank - 1
    if loss_name == "tdt":
        num_classes -= model.joint.num_extra_outputs
    model.loss = RNNTLoss(
        num_classes=num_classes,
        loss_name=loss_name,
        loss_kwargs=loss_kwargs,
        reduction=model.cfg.get("rnnt_reduction", "mean_batch"),
    )
    if model.joint.fuse_loss_wer or (
        model.decoding.joint_fused_batch_size is not None and model.decoding.joint_fused_batch_size > 0
    ):
        model.joint.set_loss(model.loss)
        model.joint.set_wer(model.wer)
    print(f"Loss: {loss_name}, {num_classes} classes, {model.joint.num_extra_outputs} extra outputs")


def check_tdt_setup(model) -> None:
    """Stop before any training if a TDT model isn't set up as TDT."""
    loss_name, _ = model.extract_rnnt_loss_cfg(model.cfg.get("loss", None))
    if loss_name != "tdt":
        return
    expected = model.joint.num_classes_with_blank - 1 - model.joint.num_extra_outputs
    implementation = model.loss._loss.__class__.__name__
    if "TDT" not in implementation or model.loss._loss.blank != expected:
        raise SystemExit(
            f"TDT setup is wrong: loss is {implementation}, blank "
            f"{model.loss._loss.blank}, expected {expected}. Stopping before training."
        )


def layerwise_param_groups(model, base_lr: float, decay: float) -> list[dict]:
    """Lower learning rates toward the input. The decoder and joint get the full rate."""
    layers = list(model.encoder.layers)
    groups = [{"params": list(model.encoder.pre_encode.parameters()), "lr": base_lr * decay ** (len(layers) + 1)}]
    for index, layer in enumerate(layers):
        groups.append({"params": list(layer.parameters()), "lr": base_lr * decay ** (len(layers) - index)})
    known = {id(p) for group in groups for p in group["params"]}
    groups.append({"params": [p for p in model.parameters() if id(p) not in known], "lr": base_lr})
    return [g for g in groups if g["params"]]


class FrozenEncoderEval(Callback):
    """Keep a frozen encoder in eval mode, since Lightning switches everything to train mode."""

    def on_train_epoch_start(self, trainer, pl_module) -> None:
        pl_module.encoder.eval()


class CollapseGuard(Callback):
    """Stop paying for a run whose model has stopped learning to transcribe.

    A public Parakeet fine-tune collapsed into repeating one token at 109.5% WER
    after a tokenizer swap. WER over the limit after the patience runs out means
    the same failure, so the run stops instead of burning GPU hours.
    """

    def __init__(self, limit: float, patience: int, metric: str = "val_wer"):
        self.limit, self.patience, self.metric, self.seen = limit, patience, metric, 0

    def on_validation_end(self, trainer, pl_module) -> None:
        if trainer.sanity_checking or self.metric not in trainer.callback_metrics:
            return
        self.seen += 1
        wer = float(trainer.callback_metrics[self.metric]) * 100
        print(f"validation {self.seen}: {self.metric} {wer:.2f}%")
        if self.seen >= self.patience and wer >= self.limit:
            print(f"CollapseGuard: {wer:.1f}% after {self.seen} validations. Stopping.")
            trainer.should_stop = True


class LossLog(Callback):
    """Print training loss, because a flat WER alone can't separate slow from broken."""

    def __init__(self, every: int):
        self.every = every

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx) -> None:
        if trainer.global_step % self.every == 0 and isinstance(outputs, dict) and "loss" in outputs:
            print(f"step {trainer.global_step}: loss {float(outputs['loss']):.3f}", flush=True)


def data_config(manifest: str, data: dict, *, shuffle: bool) -> dict:
    return {
        "manifest_filepath": manifest,
        "sample_rate": 16000,
        "batch_size": data["batch_size"],
        "shuffle": shuffle,
        "num_workers": data.get("num_workers", 2),
        "pin_memory": False,
        "use_start_end_token": False,
        "trim_silence": False,
        "max_duration": data.get("max_duration", 20.0),
        "min_duration": 0.1,
    }


def load_model(stage: str, config: dict, output: Path):
    if stage == "head":
        base = config["base_model"]
        model = ASRModel.restore_from(base, map_location="cpu") if Path(base).exists() else ASRModel.from_pretrained(base, map_location="cpu")
        model.change_vocabulary(new_tokenizer_dir=config["tokenizer_dir"], new_tokenizer_type="bpe")
        restore_tdt_loss(model)
        model.encoder.freeze()
    else:
        start = config.get("init_from") or str(output / "head.nemo")
        model = ASRModel.restore_from(start, map_location="cpu")
        restore_tdt_loss(model)
        model.encoder.unfreeze()
    check_tdt_setup(model)
    return model


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("config")
    parser.add_argument("--stage", required=True, choices=["head", "full"])
    parser.add_argument("--resume", help="Lightning checkpoint to resume from")
    parser.add_argument("--max-seconds", type=int, help="Stop cleanly after this much wall-clock time")
    args = parser.parse_args()

    config = yaml.safe_load(Path(args.config).read_text(encoding="utf-8"))
    stage = config["stages"][args.stage]
    output = Path(config["output_dir"])
    output.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(config.get("seed", 0))
    model = load_model(args.stage, config, output)
    if not config.get("spec_augment", True):
        model.spec_augmentation = None

    model.setup_training_data(OmegaConf.create(data_config(config["train_manifest"], config["data"], shuffle=True)))
    model.setup_validation_data(OmegaConf.create(data_config(config["dev_manifest"], config["data"], shuffle=False)))
    with open_dict(model.cfg):
        model.cfg.train_ds = OmegaConf.create(data_config(config["train_manifest"], config["data"], shuffle=True))
        model.cfg.validation_ds = OmegaConf.create(data_config(config["dev_manifest"], config["data"], shuffle=False))

    if args.stage == "full" and stage.get("layer_decay"):
        groups = layerwise_param_groups(model, stage["lr"], stage["layer_decay"])
        model.setup_optimizer_param_groups = lambda: setattr(model, "_optimizer_param_groups", groups)

    # A constant or cosine schedule, since the stock warmup can reset to a high rate.
    model.setup_optimization(OmegaConf.create({
        "name": "adamw",
        "lr": stage["lr"],
        "weight_decay": stage.get("weight_decay", 1e-3),
        "betas": [0.9, 0.98],
        "sched": {
            "name": "CosineAnnealing",
            "warmup_steps": stage.get("warmup_steps", 0),
            "min_lr": stage.get("min_lr", 1e-6),
            "max_steps": stage["max_steps"],
        },
    }))

    callbacks = [
        CollapseGuard(config["collapse_guard"]["wer_limit"], config["collapse_guard"]["patience"]),
        LossLog(stage.get("log_every", 10)),
        ModelCheckpoint(dirpath=output / f"checkpoints_{args.stage}", save_last=True, every_n_train_steps=stage["checkpoint_every"], save_top_k=0),
    ]
    if args.stage == "head":
        callbacks.append(FrozenEncoderEval())

    trainer = pl.Trainer(
        accelerator=config.get("accelerator", "auto"),
        devices=1,
        precision=config.get("precision", "32-true"),
        max_steps=stage["max_steps"],
        max_epochs=-1,
        max_time={"seconds": args.max_seconds} if args.max_seconds else None,
        val_check_interval=None,
        check_val_every_n_epoch=stage.get("validate_every_epochs", 1),
        log_every_n_steps=stage.get("log_every", 10),
        gradient_clip_val=1.0,
        accumulate_grad_batches=config["data"].get("accumulate", 1),
        callbacks=callbacks,
        enable_progress_bar=False,
        logger=False,
    )
    model.set_trainer(trainer)
    trainer.fit(model, ckpt_path=args.resume)

    path = output / f"{args.stage}.nemo"
    model.save_to(str(path))
    print(f"Saved {path}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/kalaris-stt/configs/datasets.yaml
# Dataset registry. Nothing is used for training until `status: confirmed`.
#
# status:
#   confirmed       A human read the dataset's own license and approved this use
#   listed          License is tagged on the source page; not yet reviewed by a human
#   secondary       License known only from a secondary source
#   excluded        Do not use
#
# access: open | gated (a human must accept terms) | unknown

datasets:
  # Wave 1: Hindi
  - name: IndicVoices
    languages: [hi, multi]
    source: https://huggingface.co/datasets/ai4bharat/IndicVoices
    license: CC-BY-4.0
    status: listed
    access: gated
    use: [train, dev]
  - name: Kathbath
    languages: [hi, multi]
    source: https://huggingface.co/datasets/ai4bharat/Kathbath
    license: CC-BY-4.0
    status: listed
    access: gated
    use: [train, dev]
  - name: Shrutilipi
    languages: [hi, multi]
    source: https://huggingface.co/datasets/ai4bharat/Shrutilipi
    license: CC-BY-4.0
    status: listed
    access: gated
    use: [train]
    notes: All India Radio broadcast speech, so check for speaker diversity.
  - name: Vistaar benchmark sets
    languages: [hi, multi]
    source: https://arxiv.org/abs/2305.15386
    license: unknown
    status: secondary
    access: unknown
    use: [test]
    notes: Inherits terms from each underlying corpus. Check them set by set.
  - name: FLEURS
    languages: [hi, en, multi]
    source: https://huggingface.co/datasets/google/fleurs
    license: CC-BY-4.0
    status: confirmed
    confirmed: Approved by the owner for training on 2026-11-08.
    access: open
    use: [train, dev, test]
    notes: Phase 0 Hindi baseline. Small and read aloud, so it can't stand in for conversational Hindi.
  - name: Common Voice 17.0
    languages: [hi, en, multi]
    source: https://huggingface.co/datasets/mozilla-foundation/common_voice_17_0
    license: CC0-1.0
    status: secondary
    access: unknown
    use: [train]
    notes: Crowdsourced, so transcripts need auditing.

  # Wave 1: Hinglish. No data cleared for commercial training has been found yet.
  # This is the top data risk.
  - name: HiACC
    languages: [hinglish]
    source: https://doi.org/10.5281/zenodo.15551669
    license: CC-BY-NC-4.0
    status: listed
    access: open
    use: [test]
    notes: Real code-switched speech, 5.24 hours and 44 speakers. Non-commercial, so evaluation only.
  - name: sonexis hinglish conversations
    languages: [hinglish]
    source: https://huggingface.co/datasets/sonexis-ai/hinglish-code-switched-conversations-v1
    license: CC-BY-NC-4.0
    status: excluded
    access: gated
    use: []
    notes: Non-commercial sample with undisclosed provenance.
  - name: addyo07 noisy hinglish
    languages: [hinglish]
    source: https://huggingface.co/datasets/addyo07/noisy-hinglish-asr
    license: Apache-2.0 on the tag
    status: secondary
    access: unknown
    use: []
    notes: Provenance unknown. Don't use it until its sources are traced.
  - name: MUCS 2021 Hinglish
    languages: [hinglish]
    source: https://www.openslr.org/104/
    mirror: https://huggingface.co/datasets/dianavdavidson/MUCS-Hinglish
    license: CC-BY-SA-4.0
    status: confirmed
    confirmed: Approved by the owner for training on 2026-11-08. Share-alike terms recorded for a later legal check.
    access: open
    use: [train, dev]
    notes: >-
      About 90 hours of technical spoken tutorials mixing Hindi and English (OpenSLR SLR104).
      The Hugging Face mirror is tagged CC BY 4.0, but the original is CC BY-SA 4.0, which governs.
      Its speaker_id is a tutorial ID, not a confirmed person.
  - name: Hinglish training data
    languages: [hinglish]
    source: TBD
    license: TBD
    status: secondary
    access: unknown
    use: [train]
    notes: Placeholder. Likely needs licensing or our own collection.

  # Wave 1: English
  - name: LibriSpeech test-clean
    languages: [en]
    source: https://www.openslr.org/12
    license: CC-BY-4.0
    status: listed
    access: open
    use: [test]

  - name: LibriSpeech train
    languages: [en]
    source: https://huggingface.co/datasets/openslr/librispeech_asr
    license: CC-BY-4.0
    status: confirmed
    confirmed: Approved by the owner for training on 2026-11-08.
    access: open
    use: [train, dev]

  # Benchmarks. See configs/benchmarks.yaml. Never train on these.
  - name: Monsoon hi-IN
    languages: [hi]
    source: https://huggingface.co/datasets/VoiceArena/MonsoonASR-Open-ASR-leaderboard-hi-IN
    license: CC-BY-4.0
    status: listed
    access: open
    use: [test]
  - name: Monsoon en-IN
    languages: [en]
    source: https://huggingface.co/datasets/VoiceArena/MonsoonASR-Open-ASR-leaderboard-en-IN
    license: CC-BY-4.0
    status: listed
    access: open
    use: [test]
  - name: Lahaja
    languages: [hi]
    source: https://huggingface.co/datasets/ai4bharat/Lahaja
    license: CC-BY-4.0 in the card text, MIT in the tag
    status: secondary
    access: gated
    use: [test]
    notes: Multi-accent Hindi, 12.5 hours and 132 speakers.
  - name: Vaani
    languages: [hi, multi]
    source: https://huggingface.co/datasets/ARTPARK-IISc/Vaani
    license: CC-BY-4.0
    status: listed
    access: gated
    use: [train]
    notes: Not yet audited. Check how much of it has transcripts.

  # Excluded
  - name: nvidia Granary-v2
    languages: [multi]
    source: https://huggingface.co/datasets/nvidia/Granary-v2
    license: OpenMDW-1.1 labels; source audio keeps its own license
    status: listed
    access: open
    use: []
    notes: Labels only. Its Hindi comes from Common Voice 26 and FLEURS, so it adds cleaner labels, not new audio.
  - name: aman-hf indic_asr
    languages: [hi]
    source: https://huggingface.co/datasets/aman-hf/indic_asr
    license: CC-BY-4.0 on the tag
    status: excluded
    access: open
    use: []
    notes: Aggregates about 10,000 hours with no named sources, so upstream licenses can't be checked.
  - name: Kaggle Hindi speech datasets
    languages: [hi]
    source: https://www.kaggle.com/datasets
    license: mixed
    status: excluded
    access: open
    use: []
    notes: Searched 2026-11-08. Results were non-commercial, re-uploads without provenance, or text only.

  - name: Krutrim IndicST
    languages: [multi]
    source: https://huggingface.co/datasets/krutrim-ai-labs/IndicST
    license: conflicting
    status: excluded
    access: unknown
    use: []
    notes: One page gives the Krutrim Community License and the dataset card says TBD.

models:
  - name: Parakeet TDT 0.6B v3
    source: https://huggingface.co/nvidia/parakeet-tdt-0.6b-v3
    license: CC-BY-4.0
    status: listed
    access: open
    use: [base, baseline]
  - name: IndicConformer 600M
    source: https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual
    license: MIT
    status: listed
    access: gated
    use: [base, baseline]
  - name: IndicConformer 600M ONNX repack (CTC only)
    source: https://huggingface.co/christopherthompson81/indicconformer-600m-onnx
    license: MIT
    status: listed
    access: open
    use: [baseline]
    notes: Third-party repack. Use it for early scores and the official model for decisions.
  - name: Whisper large-v3-turbo (MLX)
    source: https://huggingface.co/mlx-community/whisper-large-v3-turbo
    license: unknown
    status: listed
    access: open
    use: [baseline]
    notes: The MLX repo has no license tag. Upstream OpenAI Whisper is MIT.
  - name: Underdog Bark 3.0 family
    source: https://huggingface.co/ConwayResearch/Underdog-Bark-3.0
    license: CC-BY-NC-4.0
    status: excluded
    access: open
    use: []
    notes: Non-commercial. Compare against published numbers until internal benchmarking is cleared.

In [ ]:
%%writefile /kaggle/working/kalaris-stt/configs/benchmarks.yaml
# Benchmarks we report against. Selection follows the Hugging Face Open ASR
# Leaderboard, so our numbers can be compared with its table.
# Source of track choices: https://github.com/huggingface/open_asr_leaderboard
#
# Never train on or tune against anything here. `stt-eval worst` refuses these.

primary:
  hi_monsoon:
    track: Multilingual tab, Hindi column
    repo: VoiceArena/MonsoonASR-Open-ASR-leaderboard-hi-IN
    config: default
    split: test
    lattice_column: lattice
    speaker_column: speaker_id
    language: hi
    metric: oiwer
    license: CC-BY-4.0
    access: open
    notes: Conversational phone audio; 1.33 hours public and 4.47 hours private.

  en_monsoon:
    track: Main tab, part of the average
    repo: VoiceArena/MonsoonASR-Open-ASR-leaderboard-en-IN
    config: default
    split: test
    speaker_column: speaker_id
    language: en
    metric: wer
    license: CC-BY-4.0
    access: open
    notes: Indian English; 5.62 hours public. Check its text column on first fetch.

# The other main-tab sets. Fetch them when disk allows.
english_average:
  - {name: ami_cleaned, repo: hf-audio/open-asr-leaderboard, config: ami_cleaned, split: test}
  - {name: gigaspeech_cleaned, repo: hf-audio/open-asr-leaderboard, config: gigaspeech_cleaned, split: test}
  - {name: spgispeech, repo: hf-audio/open-asr-leaderboard, config: spgispeech, split: test}
  - {name: voxpopuli_cleaned_aa, repo: hf-audio/open-asr-leaderboard, config: voxpopuli_cleaned_aa, split: test}
  - {name: urgent2024, repo: hf-audio/open-asr-leaderboard, config: urgent2024_clean, split: test}
  - {name: earnings22_chunked, repo: ArtificialAnalysis/Earnings22-Cleaned-AA-chunked, config: default, split: test}

secondary:
  librispeech_clean:
    repo: hf-audio/open-asr-leaderboard
    config: librispeech
    split: test.clean
    language: en
    metric: wer
    license: CC-BY-4.0
    access: open
    notes: Benchmark-fitting check on the leaderboard, not in its current average.
  fleurs_hi:
    repo: google/fleurs
    config: hi_in
    split: test
    language: hi
    metric: wer
    license: CC-BY-4.0
    access: open
    notes: Read speech, and it has no speaker IDs.
  lahaja:
    repo: ai4bharat/Lahaja
    split: test
    language: hi
    metric: wer
    license: CC-BY-4.0 in the card text, MIT in the tag
    access: gated
    notes: 12.5 hours, 132 speakers, multi-accent. Best extra Hindi check.
  hiacc:
    repo: https://doi.org/10.5281/zenodo.15551669
    language: hinglish
    metric: wer
    license: CC-BY-NC-4.0
    access: open
    notes: Not on Hugging Face. Real code-switched speech, evaluation and research use only.

# Published results are snapshotted by scripts/snapshot_leaderboard.py into
# reports/leaderboard_snapshot.md.

In [ ]:
%%writefile /kaggle/working/kalaris-stt/training/configs/v0.yaml
# First Kalaris STT model: Hindi, Hinglish and English, from open data only.
# Licenses were approved by the owner on 2026-11-08 (see configs/datasets.yaml).
# Runs on a free Kaggle GPU through training/kaggle/kalaris_v0.ipynb.
#
# This is a starting model to prove the full pipeline on real data. It has
# little pure Hindi (about 7 hours), so expect it to be strongest on Hinglish.
name: v0
sources:
  - name: mucs_hinglish
    registry: MUCS 2021 Hinglish
    repo: dianavdavidson/MUCS-Hinglish
    config: default
    split: train
    text_column: transcript
    speaker_column: speaker_id     # A tutorial ID, used as the closest speaker proxy
    language: hinglish
    license: CC-BY-SA-4.0
    dev_fraction: 0.03
  - name: fleurs_hi
    registry: FLEURS
    repo: google/fleurs
    config: hi_in
    split: train
    text_column: transcription     # The normalized column: lower case, no punctuation
    speaker_column: speaker_id
    language: hi
    license: CC-BY-4.0
    repeat: 2                      # Oversampled, because pure Hindi is scarce here
  - name: fleurs_hi_dev
    registry: FLEURS
    repo: google/fleurs
    config: hi_in
    split: validation
    text_column: transcription
    language: hi
    license: CC-BY-4.0
    role: dev
  - name: librispeech_en
    registry: LibriSpeech train
    repo: openslr/librispeech_asr
    config: clean
    split: train.100
    text_column: text
    speaker_column: speaker_id
    language: en
    license: CC-BY-4.0
    lowercase: true
    max_hours: 50                  # English keeps its skill without crowding out Hindi
  - name: librispeech_en_dev
    registry: LibriSpeech train
    repo: openslr/librispeech_asr
    config: clean
    split: validation
    text_column: text
    language: en
    license: CC-BY-4.0
    lowercase: true
    role: dev
    max_hours: 0.5

limits:
  min_seconds: 0.5
  max_seconds: 20.0
  min_chars_per_second: 2
  max_chars_per_second: 40

base_model: nvidia/parakeet-tdt-0.6b-v3
tokenizer_dir: /kaggle/temp/tokenizers/v0
train_manifest: /kaggle/temp/training/v0/train.json
dev_manifest: /kaggle/temp/training/v0/dev.json
output_dir: /kaggle/working/v0
accelerator: gpu
precision: 16-mixed      # Kaggle's T4 and P100 have no bf16
seed: 0
spec_augment: true

data:
  batch_size: 8
  num_workers: 4
  max_duration: 20.0
  accumulate: 4          # 32 clips per optimizer step

collapse_guard:
  wer_limit: 100
  patience: 4

stages:
  head:
    lr: 0.0005
    max_steps: 3000
    warmup_steps: 300
    checkpoint_every: 500
    validate_every_epochs: 1
    log_every: 50
  full:
    lr: 0.00005
    layer_decay: 0.95
    max_steps: 20000
    warmup_steps: 300
    checkpoint_every: 500
    validate_every_epochs: 1
    log_every: 50

## Train

In [ ]:
# Point the config at this machine's folders.
import yaml
config = yaml.safe_load((REPO / "training/configs/v0.yaml").read_text())
config["tokenizer_dir"] = str(DATA / "tokenizers/v0")
config["train_manifest"] = str(DATA / "training/v0/train.json")
config["dev_manifest"] = str(DATA / "training/v0/dev.json")
config["output_dir"] = "/kaggle/working/v0"
if HEAD_FROM:
    config["init_from"] = HEAD_FROM
(REPO / "training/configs/v0_kaggle.yaml").write_text(yaml.safe_dump(config, allow_unicode=True))
os.environ["PYTHONPATH"] = str(REPO)
os.environ["PYTHONWARNINGS"] = "ignore"

In [ ]:
# Download and filter the approved data. MUCS is about 9 GB, so this takes a while.
sh(f"python training/prepare_data.py training/configs/v0_kaggle.yaml --output-root {DATA / 'training'}")
print((DATA / "training/v0/report.txt").read_text())

In [ ]:
# One tokenizer for Devanagari, Latin and later-wave scripts.
sh(f"python training/train_tokenizer.py --manifest {DATA / 'training/v0/train.json'} "
   f"--output {DATA / 'tokenizers/v0'} --vocab-size 2048 "
   "--extra-fleurs bn_in,mr_in,te_in,ta_in,gu_in,kn_in,ml_in,pa_in,or_in")

In [ ]:
# Stage 1: new tokenizer, decoder and joint; encoder frozen.
if not SKIP_HEAD:
    head_seconds = int(min(2.0, hours_left() / 4) * 3600)
    sh(f"python training/finetune.py training/configs/v0_kaggle.yaml --stage head --max-seconds {head_seconds}")

In [ ]:
# Stage 2: everything trains. Stops cleanly before Kaggle's limit and saves full.nemo.
full_seconds = int((hours_left() - 0.75) * 3600)
assert full_seconds > 1800, "Too little time left. Resume in a new session."
resume = f" --resume {RESUME}" if RESUME else ""
sh(f"python training/finetune.py training/configs/v0_kaggle.yaml --stage full --max-seconds {full_seconds}{resume}")

## Dev scores and outputs

In [ ]:
# Dev scores only (no benchmark sets). Hindi, Hinglish and English are scored separately.
scores = {}
for language, scored_as in [("hi", "hi"), ("hinglish", "hinglish"), ("en", "en")]:
    manifest = DATA / f"training/v0/dev_{language}.json"
    if not manifest.exists():
        continue
    out = Path(f"/kaggle/working/v0/dev_{language}.jsonl")
    sh(f"python -m stt_eval.cli transcribe --backend nemo --model /kaggle/working/v0/full.nemo "
       f"--manifest {manifest} --out {out} --language {scored_as}")
    result = subprocess.run(
        [sys.executable, "-m", "stt_eval.cli", "score", "--manifest", str(manifest), "--hypotheses", str(out), "--language", scored_as],
        cwd=REPO, capture_output=True, text=True, check=True)
    scores[language] = json.loads(result.stdout)
    print(language, round(scores[language]["wer"]["rate"], 2), "WER on", scores[language]["clips"], "dev clips")
Path("/kaggle/working/v0/dev_scores.json").write_text(json.dumps(scores, indent=2, ensure_ascii=False))

In [ ]:
# Keep the tokenizer and data report with the model, and drop older checkpoints to stay under 20 GB.
shutil.copytree(DATA / "tokenizers/v0", "/kaggle/working/v0/tokenizer", dirs_exist_ok=True)
shutil.copy(DATA / "training/v0/report.txt", "/kaggle/working/v0/data_report.txt")
for stage in ("head", "full"):
    folder = Path(f"/kaggle/working/v0/checkpoints_{stage}")
    for checkpoint in folder.glob("*.ckpt") if folder.exists() else []:
        if checkpoint.name != "last.ckpt":
            checkpoint.unlink()
print(subprocess.run("du -sh /kaggle/working/v0/*", shell=True, capture_output=True, text=True).stdout)